# Notebook 02: Preprocessing and Data Pipeline

This notebook prepares the raw data for model training. We will handle missing values, standardize numerical columns, one-hot encode categorical features, and split the data.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## 1. Load Data and Handle Missing Values

We convert `TotalCharges` to numeric. Missing values in `TotalCharges` represent new customers (tenure=0). We will fill these nulls with 0.0.

In [2]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Null values in TotalCharges before imputation: {df["TotalCharges"].isnull().sum()}')

# Fill missing values with 0 since tenure is 0 for these rows
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
print(f'Null values in TotalCharges after imputation: {df["TotalCharges"].isnull().sum()}')

Null values in TotalCharges before imputation: 11
Null values in TotalCharges after imputation: 0


## 2. Map Target Variable

We map the binary target column `Churn` ('Yes'/'No') to 1/0.

In [3]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Target Churn distribution:')
print(df['Churn'].value_counts(normalize=True))

Target Churn distribution:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


## 3. Train-Test Split

We split the dataset into 80% train and 20% test sets, using stratification to maintain the distribution of the target class.

In [4]:
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')

Train shape: (5634, 19)
Test shape: (1409, 19)


## 4. Construct ColumnTransformer Pipeline

We scale numerical variables (`tenure`, `MonthlyCharges`, `TotalCharges`) and one-hot encode categorical variables, dropping the first category to avoid multicollinearity.

In [5]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

# Fit-transform on training and transform on testing
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f'Preprocessed Train shape: {X_train_preprocessed.shape}')
print('Feature Names:')
print(list(feature_names))

Preprocessed Train shape: (5634, 30)
Feature Names:
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender_Male', 'cat__SeniorCitizen_1', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__PhoneService_Yes', 'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_No internet service', 'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes', 'cat__DeviceProtection_No internet service', 'cat__DeviceProtection_Yes', 'cat__TechSupport_No internet service', 'cat__TechSupport_Yes', 'cat__StreamingTV_No internet service', 'cat__StreamingTV_Yes', 'cat__StreamingMovies_No internet service', 'cat__StreamingMovies_Yes', 'cat__Contract_One year', 'cat__Contract_Two year', 'cat__PaperlessBilling_Yes', 'cat__PaymentMethod_Credit card (automatic)', 'cat__PaymentMethod_Electronic check', 'cat__PaymentMethod_Mailed check']


## 5. Save the Preprocessor and Splits

We save the preprocessor object and the train/test splits so they can be loaded in the training notebook.

In [6]:
os.makedirs('../models', exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump((X_train, X_test, y_train, y_test), '../data/split_data.pkl')
print('Successfully saved preprocessor and data splits.')

Successfully saved preprocessor and data splits.
